# DermaScan — Treinamento do Modelo
Treina um EfficientNet-B3 no dataset ISIC 2019 para classificar lesões de pele.

**Requisitos:** Google Colab com GPU ativada (Runtime → Change runtime type → GPU)
**Tempo estimado:** ~1h (20 epochs)

In [ ]:
# Instala dependências
!pip install -q torch torchvision efficientnet-pytorch pandas scikit-learn matplotlib seaborn tqdm kagglehub

# Monta Google Drive (pra salvar o modelo)
from google.colab import drive
drive.mount('/content/drive')

import os
import matplotlib.pyplot as plt

os.makedirs('/content/drive/MyDrive/dermascan', exist_ok=True)

In [ ]:
import os

REPO_URL = 'https://github.com/somoza00/dermascan.git'
REPO_DIR = '/content/dermascan'

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

%cd {REPO_DIR}/dermascan-model


## 1. Baixar Dataset ISIC 2019

Usa o kagglehub pra baixar o dataset ISIC 2019 do Kaggle.

**O dataset ISIC 2019 contém:**
- ~25.000 imagens de lesões de pele
- 8 classes: Melanoma, Carcinoma Basocelular, Nevo Benigno, Ceratose Seborreica, etc.
- Imagens dermatoscópicas padronizadas

In [ ]:
# Baixa ISIC 2019 do Kaggle via kagglehub
import kagglehub

print('Baixando dataset ISIC 2019...')
path = kagglehub.dataset_download('andrewmvd/isic-2019')
print(f'Dataset em: {path}')

!ls "{path}" | head -20

In [ ]:
# Exploração básica dos dados
import os

import pandas as pd

# Busca recursiva: o kagglehub costuma aninhar os arquivos em subpastas
# (ex.: "ISIC_2019_Training_Input/", CSVs dentro de uma pasta versionada) —
# listar só o nível raiz de `path` (como antes) podia não achar nada.
csv_files = []
for root, _, files in os.walk(path):
    for f in files:
        if f.endswith('.csv'):
            csv_files.append(os.path.join(root, f))
print(f'CSVs encontrados: {csv_files}')

# O ground truth oficial do ISIC 2019 vem em formato one-hot (uma coluna
# 0/1 por classe), não numa coluna `dx` única — por isso tem que ser
# especificamente o CSV de ground truth, não o de metadata (idade/sexo/
# local da lesão) nem qualquer CSV "primeiro da lista". Preferimos a
# variante "DuplicateRemoved" quando existir: remove 2 entradas duplicadas
# problemáticas dos dados oficiais do challenge.
ground_truth_candidates = [f for f in csv_files if 'groundtruth' in os.path.basename(f).lower()]
if not ground_truth_candidates:
    raise FileNotFoundError(
        f'Nenhum CSV de ground truth encontrado em {path}. CSVs disponíveis: {csv_files}'
    )
csv_path = sorted(
    ground_truth_candidates,
    key=lambda f: 'duplicateremoved' not in os.path.basename(f).lower(),
)[0]
print(f'Usando ground truth: {csv_path}')

df = pd.read_csv(csv_path)
print(f'Shape: {df.shape}')
print(f'Colunas: {df.columns.tolist()}')

# A coluna de id vem como "image" no CSV oficial (ex.: "ISIC_0024306", sem
# extensão) — `SkinLesionDataset` espera a coluna "image_id".
if 'image' in df.columns and 'image_id' not in df.columns:
    df = df.rename(columns={'image': 'image_id'})

# Converte o one-hot pra uma coluna `dx` categórica única, que é o que
# `SkinLesionDataset`, o split estratificado e `src/risk.py` esperam.
# Ignora deliberadamente a coluna "UNK": ela existe no formato do challenge
# pra imagens fora de distribuição e não é usada no conjunto de treino —
# incluí-la criaria uma 9ª classe fantasma, incompatível com as 8 classes
# fixas de `src/risk.py::CLASS_NAMES`.
if 'dx' not in df.columns:
    from src.risk import CLASS_NAMES

    label_cols = [c for c in CLASS_NAMES if c in df.columns]
    missing_classes = set(CLASS_NAMES) - set(label_cols)
    if missing_classes:
        raise ValueError(f'CSV não contém as colunas esperadas: {sorted(missing_classes)}')

    unlabeled_mask = df[label_cols].sum(axis=1) == 0
    if unlabeled_mask.any():
        print(f'⚠️  {int(unlabeled_mask.sum())} imagens sem nenhuma classe conhecida marcada '
              '(provavelmente UNK) — removidas.')
        df = df[~unlabeled_mask].reset_index(drop=True)

    df['dx'] = df[label_cols].idxmax(axis=1)

print(f'\nDistribuição das classes:')
print(df['dx'].value_counts())
print(f'\nAmostras:')
print(df.head())

# As imagens ficam numa subpasta (ex.: "ISIC_2019_Training_Input/"), não
# direto na raiz retornada pelo kagglehub — acha a pasta com mais .jpg em
# vez de assumir `path` como diretório de imagens.
image_dirs = {}
for root, _, files in os.walk(path):
    jpg_count = sum(1 for f in files if f.lower().endswith('.jpg'))
    if jpg_count:
        image_dirs[root] = jpg_count
if not image_dirs:
    raise FileNotFoundError(f'Nenhuma imagem .jpg encontrada em {path}')
img_dir = max(image_dirs, key=image_dirs.get)
print(f'\nPasta de imagens: {img_dir} ({image_dirs[img_dir]} arquivos .jpg)')

In [ ]:
# Split treino/validação/teste
from sklearn.model_selection import train_test_split

# Estratificado em 3 partes. O conjunto de teste fica isolado até a
# avaliação final (cell 12) — o de validação é usado tanto pra early
# stopping quanto pra escolher o "melhor" checkpoint durante o treino
# (cell 10), então ele sozinho não pode ser a única fonte da métrica final:
# senão a avaliação fica artificialmente otimista (mede o modelo nos
# próprios dados que ajudaram a escolhê-lo).
train_df, temp_df = train_test_split(df, test_size=0.30, random_state=42, stratify=df['dx'])
val_df, test_df = train_test_split(temp_df, test_size=0.50, random_state=42, stratify=temp_df['dx'])

print(f'Treino: {len(train_df)} amostras')
print(f'Validação: {len(val_df)} amostras')
print(f'Teste (holdout, só usado na avaliação final): {len(test_df)} amostras')

print('\nDistribuição treino:')
print(train_df['dx'].value_counts(normalize=True))

# Classes fixas, calculadas UMA VEZ a partir do dataset completo (antes do
# split) e reutilizadas nos 3 datasets (cell 7). Se cada split calculasse
# seu próprio `sorted(df['dx'].unique())`, um split que por azar não
# contivesse uma classe rara (DF/VASC têm <1% do ISIC 2019) desalinharia
# os índices entre treino e validação — silenciosamente, sem nenhum erro,
# e sem garantia de que o índice de MEL é o mesmo nos dois.
CLASSES = sorted(df['dx'].unique())
print(f'\nClasses (ordem fixa = índice de saída do modelo): {CLASSES}')

# Trava de sanidade: se o CSV baixado não contiver as 8 classes exatas que
# src/risk.py espera (ex.: uma classe rara ausente por acaso na amostra),
# o modelo treinaria com um número de saídas incompatível com a regra de
# negócio — melhor falhar aqui do que descobrir isso na hora de plugar na API.
from src.risk import CLASS_NAMES as EXPECTED_CLASSES

if CLASSES != EXPECTED_CLASSES:
    raise ValueError(
        f'Classes do dataset ({CLASSES}) não batem com as 8 classes fixas '
        f'esperadas por src/risk.py ({EXPECTED_CLASSES}).'
    )

## 2. Criar Dataset e Modelo

**Por que EfficientNet-B3?**
- Melhor relação accuracy/parametros que ResNet ou VGG
- B3 é um bom equilíbrio entre performance e tamanho (1536 features)
- Pré-treinado no ImageNet — transfere conhecimento de formas gerais

In [ ]:
import numpy as np
import torch
from torch.utils.data import DataLoader

from src.dataset import SkinLesionDataset, build_transform, IMAGE_SIZE

# Seeds — sem isso, duas execuções com os "mesmos" hiperparâmetros produzem
# modelos diferentes (inicialização da head, shuffle do DataLoader),
# dificultando comparar mudanças entre runs.
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

# `build_transform` (src/dataset.py) resolve em IMAGE_SIZE=300 (resolução
# nativa do B3, não 224 do B0) preservando aspect ratio via Resize+Crop —
# um Resize direto pra um quadrado fixo distorce a forma da lesão, o que é
# clinicamente relevante (critério de assimetria do ABCDE de melanoma).
train_dataset = SkinLesionDataset(train_df, img_dir, transform=build_transform(IMAGE_SIZE, train=True), classes=CLASSES)
val_dataset = SkinLesionDataset(val_df, img_dir, transform=build_transform(IMAGE_SIZE, train=False), classes=CLASSES)
test_dataset = SkinLesionDataset(test_df, img_dir, transform=build_transform(IMAGE_SIZE, train=False), classes=CLASSES)

print(f'Classes: {train_dataset.classes}')
print(f'Treino: {len(train_dataset)} | Validação: {len(val_dataset)} | Teste: {len(test_dataset)}')

# Dataloaders
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=2)

In [ ]:
# Inicializa o modelo
from src.model import DermaScanModel
from src.train import compute_class_weights

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Dispositivo: {device}')

model = DermaScanModel(num_classes=len(train_dataset.classes))
model = model.to(device)

# Pesos por classe, inversamente proporcionais à frequência no treino. Sem
# isso, o modelo tende a "apostar" em NV (majoritária e benigna) nos casos
# ambíguos — o viés mais perigoso possível aqui, porque reduz recall
# justamente das classes raras e malignas (MEL, SCC, BCC).
class_counts = train_df['dx'].value_counts().to_dict()
class_weights = compute_class_weights(class_counts, train_dataset.classes, device)
print(f'Pesos por classe: {dict(zip(train_dataset.classes, class_weights.cpu().numpy().round(2)))}')

criterion = torch.nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-5)

# Scheduler: reduz LR quando a loss estagna
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=2, verbose=True
)

## 3. Treinamento

20 epochs com early stopping. O modelo salva automaticamente quando a loss de validação melhora.

In [ ]:
from src.train import train_one_epoch, evaluate, clinical_metrics, EarlyStopping

MODEL_PATH = '/content/drive/MyDrive/dermascan/dermascan_v1.pt'

num_epochs = 20
# mode='max': seleciona e para o treino por macro-F1, não por val_loss —
# com NV dominando o dataset, loss/accuracy ficam ótimas mesmo com recall
# ruim nas classes raras (MEL incluída). macro-F1 pondera todas as classes
# igualmente, então esconder um recall ruim de MEL fica bem mais difícil.
early_stopping = EarlyStopping(patience=5, mode='max')
best_macro_f1 = float('-inf')

history = {
    'train_loss': [], 'train_acc': [],
    'val_loss': [], 'val_acc': [], 'val_macro_f1': [], 'val_mel_recall': [],
}

for epoch in range(1, num_epochs + 1):
    print(f'\n=== Epoch {epoch}/{num_epochs} ===')

    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_acc, preds, labels = evaluate(model, val_loader, criterion, device)
    metrics = clinical_metrics(preds, labels, train_dataset.classes, critical_class='MEL')
    mel_recall = metrics.get('critical_recall', float('nan'))

    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    history['val_macro_f1'].append(metrics['macro_f1'])
    history['val_mel_recall'].append(mel_recall)

    print(f'Train Loss: {train_loss:.4f} | Acc: {train_acc:.2f}%')
    print(f'Val   Loss: {val_loss:.4f} | Acc: {val_acc:.2f}% | Macro-F1: {metrics["macro_f1"]:.4f} '
          f'| Recall MEL: {mel_recall:.4f}')

    scheduler.step(val_loss)

    # Salva o checkpoint quando macro-F1 melhora (não val_loss/accuracy)
    if metrics['macro_f1'] > best_macro_f1:
        best_macro_f1 = metrics['macro_f1']
        torch.save(model.state_dict(), MODEL_PATH)
        print(f'✅ Melhor modelo salvo! (macro-F1: {best_macro_f1:.4f}, recall MEL: {mel_recall:.4f})')

    early_stopping(metrics['macro_f1'])
    if early_stopping.early_stop:
        print('⛔ Early stopping ativado')
        break

## 4. Avaliação Final

Métricas no conjunto de validação: matriz de confusão, classification report.

In [ ]:
from src.evaluate import plot_confusion_matrix, print_metrics
from src.train import clinical_metrics

# Recarrega o melhor checkpoint (por macro-F1). O objeto `model` em memória
# ficou com os pesos do ÚLTIMO epoch treinado, não necessariamente do
# melhor — sem este reload, a avaliação abaixo mediria o modelo errado.
model.load_state_dict(torch.load(MODEL_PATH, map_location=device))

# Avaliação FINAL no conjunto de TESTE — holdout nunca visto durante o
# treino nem usado pra escolher o checkpoint (que usaram só train/val).
# Rodar isso no mesmo `val_loader` usado pra seleção infla as métricas
# reportadas, porque o modelo já foi "escolhido" pra ir bem nesses dados.
test_loss, test_acc, test_preds, test_labels = evaluate(model, test_loader, criterion, device)
test_metrics = clinical_metrics(test_preds, test_labels, train_dataset.classes, critical_class='MEL')

print(f'\nAcurácia final (teste): {test_acc:.2f}%')
print(f'Macro-F1 (teste): {test_metrics["macro_f1"]:.4f}')
print(f'Recall de MEL (teste): {test_metrics.get("critical_recall", float("nan")):.4f}  <- métrica mais importante deste modelo')

# Métricas detalhadas + destaque do recall de MEL
print_metrics(test_labels, test_preds, train_dataset.classes, critical_class='MEL')

# Matriz de confusão normalizada por linha — com NV dominando o dataset,
# uma matriz de contagens brutas esconde visualmente os erros nas classes
# raras (MEL incluída).
plot_confusion_matrix(
    test_labels, test_preds, train_dataset.classes,
    '/content/drive/MyDrive/dermascan/confusion_matrix.png',
    normalize=True,
)

In [ ]:
# Gráfico de perda/acurácia
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(history['train_loss'], label='Treino')
ax1.plot(history['val_loss'], label='Validação')
ax1.set_title('Loss')
ax1.legend()

ax2.plot(history['train_acc'], label='Treino')
ax2.plot(history['val_acc'], label='Validação')
ax2.set_title('Acurácia (%)')
ax2.legend()

plt.tight_layout()
plt.savefig('/content/drive/MyDrive/dermascan/training_history.png')
plt.show()

## 5. Exportar pra Produção

O modelo será copiado para `dermascan-api/models/` depois do treino.

In [ ]:
# Exporta com o contrato completo que a API precisa pra reconstruir o
# pipeline de inferência: pesos + ordem das classes (define o índice de
# MEL) + resolução de entrada + normalização usada no treino.
from src.evaluate import export_model

EXPORT_PATH = '/content/drive/MyDrive/dermascan/dermascan_v1.pt'
export_model(model, classes=train_dataset.classes, image_size=IMAGE_SIZE, save_path=EXPORT_PATH)

print(f'Tamanho: {os.path.getsize(EXPORT_PATH) / 1024 / 1024:.1f} MB')

print('\n📋 Próximos passos:')
print('1. Baixe o arquivo dermascan_v1.pt do Drive')
print('2. Copie para dermascan-api/models/')
print('3. Implemente um RealInferenceService em dermascan-api/app/services/inference.py')
print('   que: carrega este checkpoint, aplica build_transform(image_size, train=False)')
print('   de src/dataset.py na imagem recebida, roda softmax nos logits e usa')
print('   src/risk.py::build_prediction(probs) pra montar a resposta — mesma regra')
print('   de negócio (MEL > 30% = risco alto) usada na avaliação, um único lugar.')